# 📊 Auto Insurance Claims — Load CSV Data into Lakehouse Tables

---

> **Purpose:** Reads the 10 CSV files from the Lakehouse `Files/` area and creates
> managed Delta tables. Run this notebook **once** after uploading the CSV files
> from the `data/` folder in the repository.
>
> Each run is idempotent — existing tables are overwritten.

## ⚙️ Configuration

Set the name of the Lakehouse where you uploaded the CSV files.

In [ ]:
LAKEHOUSE_NAME = "lh_autoclaims"  # ← Change this to your Lakehouse name

## 🔍 Resolve Lakehouse Path

Locates the Lakehouse in the current workspace and builds the `abfss://` path to the `Files/` area.

In [ ]:
import sempy.fabric as fabric

WORKSPACE_ID = fabric.get_notebook_workspace_id()

client = fabric.FabricRestClient()
resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/lakehouses")
resp.raise_for_status()

lh = next(
    (l for l in resp.json().get("value", []) if l["displayName"] == LAKEHOUSE_NAME),
    None,
)
if not lh:
    raise ValueError(
        f"Lakehouse '{LAKEHOUSE_NAME}' not found in workspace. "
        f"Available: {[l['displayName'] for l in resp.json().get('value', [])]}"
    )

LAKEHOUSE_ID = lh["id"]
FILES_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Files"

print(f"✅ Lakehouse: {LAKEHOUSE_NAME}")
print(f"   ID:   {LAKEHOUSE_ID}")
print(f"   Path: {FILES_PATH}")

## 📁 Discover CSV Files

Lists all `.csv` files in the Lakehouse `Files/` area and maps each one to a table name (filename without the `.csv` extension).

In [ ]:
from notebookutils import mssparkutils

# Expected CSV files and their table names
EXPECTED_TABLES = [
    "adjuster_dim",
    "agent_dim",
    "claims_fact",
    "coverage_dim",
    "customer_dim",
    "incident_dim",
    "payments_fact",
    "policy_dim",
    "repair_shop_dim",
    "vehicle_dim",
]

# Check which CSVs are present
files = mssparkutils.fs.ls(FILES_PATH)
csv_files = {f.name.replace(".csv", ""): f.path for f in files if f.name.endswith(".csv")}

print(f"Found {len(csv_files)} CSV file(s) in Files/:")
for name in sorted(csv_files):
    status = "✅" if name in EXPECTED_TABLES else "❓"
    print(f"  {status} {name}.csv")

missing = set(EXPECTED_TABLES) - set(csv_files)
if missing:
    print(f"\n⚠️  Missing CSVs: {sorted(missing)}")
    print("   Upload them to the Lakehouse Files/ area before continuing.")
else:
    print(f"\n✅ All {len(EXPECTED_TABLES)} expected CSV files are present.")

## 🏗️ Create Managed Tables

Reads each CSV with schema inference and writes it as a managed Delta table in the Lakehouse. Tables are overwritten if they already exist.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, BooleanType, DateType

# Explicit schemas for type accuracy (inferSchema often promotes integers to longs, dates to strings)
TABLE_SCHEMAS = {
    "adjuster_dim": StructType([
        StructField("adjuster_id", StringType()),
        StructField("first_name", StringType()),
        StructField("last_name", StringType()),
        StructField("email", StringType()),
        StructField("phone", StringType()),
        StructField("license_number", StringType()),
        StructField("region", StringType()),
        StructField("specialization", StringType()),
    ]),
    "agent_dim": StructType([
        StructField("agent_id", StringType()),
        StructField("first_name", StringType()),
        StructField("last_name", StringType()),
        StructField("email", StringType()),
        StructField("phone", StringType()),
        StructField("agency_name", StringType()),
        StructField("license_number", StringType()),
        StructField("region", StringType()),
    ]),
    "claims_fact": StructType([
        StructField("claim_id", StringType()),
        StructField("claim_number", StringType()),
        StructField("policy_id", StringType()),
        StructField("customer_id", StringType()),
        StructField("vehicle_id", StringType()),
        StructField("incident_id", StringType()),
        StructField("adjuster_id", StringType()),
        StructField("repair_shop_id", StringType()),
        StructField("claim_date", DateType()),
        StructField("claim_status", StringType()),
        StructField("claim_type", StringType()),
        StructField("total_claimed_amount", DoubleType()),
        StructField("fault_indicator", StringType()),
    ]),
    "coverage_dim": StructType([
        StructField("coverage_id", StringType()),
        StructField("policy_id", StringType()),
        StructField("coverage_type", StringType()),
        StructField("coverage_limit", DoubleType()),
        StructField("deductible", DoubleType()),
        StructField("premium_portion", DoubleType()),
    ]),
    "customer_dim": StructType([
        StructField("customer_id", StringType()),
        StructField("first_name", StringType()),
        StructField("last_name", StringType()),
        StructField("date_of_birth", DateType()),
        StructField("address", StringType()),
        StructField("city", StringType()),
        StructField("state", StringType()),
        StructField("zip", IntegerType()),
        StructField("phone", StringType()),
        StructField("email", StringType()),
        StructField("driver_license_number", StringType()),
        StructField("customer_since", DateType()),
    ]),
    "incident_dim": StructType([
        StructField("incident_id", StringType()),
        StructField("incident_date", DateType()),
        StructField("incident_type", StringType()),
        StructField("location", StringType()),
        StructField("description", StringType()),
        StructField("weather_conditions", StringType()),
        StructField("police_report_number", StringType()),
        StructField("at_fault_party", StringType()),
    ]),
    "payments_fact": StructType([
        StructField("payment_id", StringType()),
        StructField("claim_id", StringType()),
        StructField("payment_date", DateType()),
        StructField("amount", DoubleType()),
        StructField("payment_type", StringType()),
        StructField("payment_method", StringType()),
        StructField("payee_name", StringType()),
        StructField("payment_status", StringType()),
    ]),
    "policy_dim": StructType([
        StructField("policy_id", StringType()),
        StructField("policy_number", StringType()),
        StructField("customer_id", StringType()),
        StructField("agent_id", StringType()),
        StructField("start_date", DateType()),
        StructField("end_date", DateType()),
        StructField("premium_amount", DoubleType()),
        StructField("policy_status", StringType()),
        StructField("policy_type", StringType()),
    ]),
    "repair_shop_dim": StructType([
        StructField("shop_id", StringType()),
        StructField("shop_name", StringType()),
        StructField("address", StringType()),
        StructField("city", StringType()),
        StructField("state", StringType()),
        StructField("phone", StringType()),
        StructField("license_number", StringType()),
        StructField("network_tier", StringType()),
        StructField("certified", BooleanType()),
    ]),
    "vehicle_dim": StructType([
        StructField("vehicle_id", StringType()),
        StructField("policy_id", StringType()),
        StructField("vin", StringType()),
        StructField("make", StringType()),
        StructField("model", StringType()),
        StructField("year", IntegerType()),
        StructField("color", StringType()),
        StructField("license_plate", StringType()),
        StructField("vehicle_type", StringType()),
        StructField("market_value", DoubleType()),
    ]),
}

created = []
failed = []

for table_name in EXPECTED_TABLES:
    if table_name not in csv_files:
        print(f"  ⏭️  Skipping {table_name} (CSV not found)")
        failed.append(table_name)
        continue

    csv_path = f"{FILES_PATH}/{table_name}.csv"
    schema = TABLE_SCHEMAS.get(table_name)

    try:
        if schema:
            df = spark.read.option("header", True).schema(schema).csv(csv_path)
        else:
            df = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)

        df.write.mode("overwrite").saveAsTable(table_name)
        row_count = df.count()
        col_count = len(df.columns)
        print(f"  ✅ {table_name:20s}  {row_count:>4} rows × {col_count} columns")
        created.append(table_name)
    except Exception as e:
        print(f"  ❌ {table_name}: {e}")
        failed.append(table_name)

print(f"\n📊 Summary: {len(created)} tables created, {len(failed)} failed")

## ✅ Verify Tables

Lists all tables in the Lakehouse and shows row counts to confirm everything loaded correctly.

In [ ]:
print(f"\n{'Table':<22} {'Rows':>6}  {'Columns':>7}")
print("─" * 40)

total_rows = 0
for table_name in sorted(EXPECTED_TABLES):
    try:
        df = spark.table(table_name)
        rows = df.count()
        cols = len(df.columns)
        total_rows += rows
        print(f"{table_name:<22} {rows:>6}  {cols:>7}")
    except Exception:
        print(f"{table_name:<22}   — not found")

print("─" * 40)
print(f"{'Total':<22} {total_rows:>6} rows across {len(created)} tables")
print(f"\n✅ Lakehouse '{LAKEHOUSE_NAME}' is ready for ontology creation.")
print(f"   Next step: Run the ontology setup notebook (autoclaims_ontology_setup.ipynb).")